In [ ]:
!pip install langchain langchain-community langchain-google-genai

In [ ]:
import os
# google AI studio
os.environ["GOOGLE_API_KEY"]="YOUR-API-KEY"

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 17.6 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    temperature = 0,
    max_tokens = 1000,
    timeout = 30,
    max_retires = 3,

)

In [ ]:
question = input("Ask something: ")
response = llm.invoke(question)
answer = response.content
if isinstance(answer,list):
  answer = "\n".join(
      item["text"]
      for item in answer
      if isinstance(item,dict) and item.get("type") == "text"
  )

print("\n"+"="*60)
print(question)
print("="*60)
print(answer)
print("="*60)

Ask something: What is Generative AI


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



What is Generative AI
**Generative AI** (short for Generative Artificial Intelligence) refers to a type of artificial intelligence system that is capable of creating **new content**—such as text, images, audio, video, code, and 3D data—based on the prompts and training data it has been given.

Unlike traditional AI, which is designed to analyze data, make predictions, or classify information (like recommending a movie or detecting spam), Generative AI creates *original* outputs that often mimic human creativity.

Here is a breakdown of how it works, what it can do, and why it matters:

---

### 1. How Does It Work?
Generative AI models are powered by **Machine Learning**, specifically deep learning neural networks. 
* **Training:** The AI is fed massive amounts of data from the internet (books, articles, artwork, photos, music). It studies this data to learn patterns, structures, and relationships (e.g., "After the word 'peanut butter,' the word 'jelly' often follows," or "A cat's ear

In [ ]:
import gradio as gr
from pypdf import PdfReader
from langchain_google_genai import ChatGoogleGenerativeAI

reader = PdfReader("ABC Restaurant.pdf")
restaurant_info=""
for page in reader.pages:
  text = page.extract_text()
  if text:
    restaurant_info += text + "\n"

def chatbot(message, history):
  prompt = f"""
You are a customer support assistant.
Use the following restaurant information
to answer the customer's question
Restaurant Information:
{restaurant_info}

Customer Question:
{message}

Answer the customer clearly and politely.
If the information is not available in the
restaurant information, say that you don't
have that information.
"""

  response = llm.invoke(prompt)

  return response.content
demo = gr.ChatInterface(
    fn = chatbot,
    title = "Restaurant Support box",
    description = "Ask questions about our restaurant"
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://125395f682cb26714e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [ ]:
loader = PyPDFLoader("/content/Vishanth V S_Resume.pdf")
documents = loader.load();

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def read_pdf(Question:str)->str:
  """Answer a question using the PDF content."""
  text = "\n".join(doc.page_content for doc in documents)
  return text

In [ ]:
agent = create_agent(
    model = llm,
    tools = [read_pdf],
    system_prompt = """
    You are a PDF reader agent.
    Use the read_pdf tool to find information from the PDF.
    Answer only using information from the PDF.
    If the answer is not available in the PDF, say:
    'I could not find this information in the PDF.'
    """
)

In [ ]:
question = input("Ask something about the PDF: ")

result = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

# Get final answer
answer = result["messages"][-1].content

# If Gemini returns a list, extract the text
if isinstance(answer, list):
    answer = "\n".join(
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    )

print("\n" + "=" * 60)
print("🤖 PDF AGENT ANSWER")
print("=" * 60)
print(answer)
print("=" * 60)

Ask something about the PDF: Give me the project details


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🤖 PDF AGENT ANSWER
Based on the provided PDF, here are the project details:

* **Project Name:** Machine Learning Model Deployment using Streamlit
* **Technologies Used:** Python, Streamlit
* **Description:** 
  * Developed and deployed a machine learning application using Streamlit.
  * Built an interactive user interface for real-time model predictions.


In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.
    Use this tool when the user asks for a mathematical calculation.
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [ ]:
calc = create_agent(
    model = llm,
    tools = [calculator],
    system_prompt = """
    You are a Calculator agent.
    Calculate the answer for the expression
    If the expression is wrong, say:
    'I could not solve this expression, provide correct expression.'
    """
)

In [ ]:
expression = input("Provide the expression")

result = calc.invoke({
    "messages": [
        {"role": "user", "content": expression}
    ]
})

# Get final answer
answer = result["messages"][-1].content

# If Gemini returns a list, extract the text
if isinstance(answer, list):
    answer = "\n".join(
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    )

print("\n" + "=" * 60)
print("🤖 Calculator AGENT ANSWER")
print("=" * 60)
print(answer)
print("=" * 60)

Provide the expression549+332


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🤖 Calculator AGENT ANSWER
881
